# CAFG-106 / HU-15: test evaluable de 149 imágenes

Restaura el checkpoint original sin entrenamiento. El test original conserva 150 imágenes (75 Roya, 75 Sanas). Se excluye únicamente Roya/Roya 752.jpg por corrupción JPEG autorizada, antes de inferencia: se evalúan 149 imágenes (74 Roya, 75 Sanas). El manifiesto original es inmutable.


In [ ]:
import os
from pathlib import Path

# En Kaggle: establecer CAFG_ML_ROOT en la carpeta ml del recurso privado adjunto.
# CAFG_IMAGE_ROOT debe apuntar directamente a coffee_leaves (contiene Roya y Sanas).
ML_ROOT = Path(os.environ.get("CAFG_ML_ROOT", "../")).resolve()
SOURCE_DIR = Path(os.environ.get("CAFG_SOURCE_DIR", str(ML_ROOT / "src"))).resolve()
MANIFEST_PATH = Path(os.environ.get("CAFG_MANIFEST", str(ML_ROOT / "reports/dataset_manifest.csv"))).resolve()
EXCLUSIONS_PATH = Path(os.environ.get("CAFG_EXCLUSIONS", str(ML_ROOT / "reports/evaluation_exclusions.json"))).resolve()
CHECKPOINT_PATH = Path(os.environ.get("CAFG_CHECKPOINT", str(ML_ROOT / "models/resnet18_coffee_base.pth"))).resolve()
IMAGE_ROOT = Path(os.environ.get("CAFG_IMAGE_ROOT", "/kaggle/input/datasets/claudiobances/coffee-leaf-dataset-by-phytosanitary-class/coffee_leaves")).resolve()
OUTPUT_DIR = Path(os.environ.get("CAFG_OUTPUT_DIR", "/kaggle/working/evaluacion-cafg-106")).resolve()

for path in [SOURCE_DIR / "dataset.py", SOURCE_DIR / "evaluation.py", MANIFEST_PATH, CHECKPOINT_PATH, EXCLUSIONS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Configura las rutas de los recursos adjuntos: {path}")
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"Configura CAFG_IMAGE_ROOT: {IMAGE_ROOT}")
print("Código:", SOURCE_DIR)
print("Imágenes:", IMAGE_ROOT)
print("Resultados:", OUTPUT_DIR)


In [ ]:
import importlib
import sys

sys.path.insert(0, str(SOURCE_DIR))
# Fallar si otra celda importó módulos homónimos desde una ubicación distinta.
for name in ("dataset", "evaluation"):
    if name in sys.modules:
        if Path(sys.modules[name].__file__).resolve() != SOURCE_DIR / f"{name}.py":
            raise RuntimeError(f"Reinicia la sesión: {name} ya se importó desde otra ubicación")
evaluation = importlib.import_module("evaluation")
print("Módulo de evaluación:", evaluation.__file__)


## Validación e inferencia

Se verifican hashes de manifiesto, checkpoint y las 150 imágenes originales. El registro permite una sola exclusión por identidad completa (ruta, SHA-256, clase, split). La inferencia usa un Subset con exactamente 149 entradas en orden original, sin decodificar el JPEG excluido. Pillow permanece en modo estricto. No modificar el registro para excluir otras muestras.


In [ ]:
metrics = evaluation.evaluate(
    manifest_path=MANIFEST_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    image_root=IMAGE_ROOT,
    output_dir=OUTPUT_DIR,
    exclusions_path=EXCLUSIONS_PATH,
    batch_size=16,
)
print(metrics)


In [ ]:
from IPython.display import Markdown, display, Image

display(Markdown((OUTPUT_DIR / "informe.md").read_text(encoding="utf-8")))
for path in sorted(OUTPUT_DIR.glob("*.png")):
    display(Image(filename=str(path)))
print("Archivos generados:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name, path.stat().st_size, "bytes")


## Revisión final

Comprobar 149 predicciones: 74 Roya y 75 Sanas. Registrar 150 originales, 1 excluida y 149 evaluadas. Revisar errores en ambas direcciones con sus denominadores efectivos, y los ejemplos visuales. La exclusión por integridad debe figurar en las limitaciones metodológicas. No presentar estos resultados como evaluación de 150 imágenes.
